# Check the SEAD source adaptation and environment.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
PROJECT_DIR = PROJECT_ROOT

PYTHON_EXECUTABLE = str(RELEASE_TORCH_PYTHON)


In [ ]:
CONFIG = {'required_hashes': {'data/train_FD001.txt': '963b5e22825b34d8b21c69e1aeb4af3e647050eb672ee8834ba4b5d91d2de0f8', 'revision_outputs/02_FD001_protocol/experiment_protocol.json': '0bd6c241f0e50cdd3c00888d62840649a06f21ca7583fb59ecffedc01979d056', 'revision_outputs/02_FD001_protocol/engine_splits.csv': '53c2ed564fcded8e502f307c0011aa1cd12ebd03b4429882e1fdf142466f7c45', 'revision_outputs/02_FD001_protocol/train_evaluation_reference.csv': 'c6b5e08384280fb687b188723fa7ecf7e76254a9ec20fc606be206f8de7a27e8', 'revision_outputs/02_FD001_protocol/role_summary.csv': 'a31fb6ead724b9190cb2a3de692c4f189f27d3cc589274c38fe65b2cffca82d7'}, 'source_audit': {'upstream_repository': 'https://github.com/pumpkinyouge/SEAD-Net', 'commit': 'edd3162d949fe1e005721c7c400c5f52852ab6ae', 'paper_doi': '10.1109/TNSM.2026.3694683', 'inspected_source_hashes': {'src/main.py': '2a118f1886978e50d0985cf59e3aaf8ef5a43607cc871a51ee0bbe7b292f1728', 'src/train.py': 'facbe0492e0bc1dda5f81d005d49285604d5897ee483abaa74bbe51e2e69bc7b', 'src/model/network.py': '93df3821675b2f3334f1e15c3cc87ba4ee5adb94ae92267b32977e91e99d0ad4', 'src/model/SEAD.py': '0ca7618de8a646673f46300713854d678f2830fd84820ad563eb31162fd5f72f', 'src/datasets.py': '84cd9c3843653b5f4edd769142519d18e6c7a2a864085a1b9a3bcd8bf665a08d', 'src/utils.py': '7bc0d318fc875bddc6aa4a5f87fc31318473c8fe5005b576c801cee96f801fa3', 'src/optimisers.py': 'f3a25ac4dfaa2aa1032086c60435fc376392de647d15106c2a069743cd7f47c6', 'config.conf': 'aa6ee1e9e74ff4a2097d1d3b7ddcf938921a8ee8ca9600843c0bf780c7e44f78', 'Supplementary Material.pdf': '6600b433be5a6da84d2337082ca68bae37b177813758d8bb714e1d061010f118'}, 'comparison_status': 'NOT_RUN', 'original_detector_count_preserved': 5, 'source_differences': ['test_set_checkpoint_selection', 'all_data_anomaly_pool_for_training_noise', 'normalized_residual_norm_in_second_stage_loss', 'L2_vs_MSE_pretrain', 'lambda_convention_reversed_in_supplement', 'runtime_epochs_differ_from_supplement', 'CUDA_specific_calls']}}

import csv
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import subprocess
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def check_sources(project, config):
    findings = []
    for relative, expected in config['required_hashes'].items():
        path = project / relative
        item = {'relative_path': relative, 'expected_sha256': expected}
        if not path.is_file():
            item['status'] = 'MISSING'
        else:
            item['sha256'] = sha256(path)
            item['status'] = 'MATCH' if item['sha256'] == expected else 'DIFFERENT'
        findings.append(item)
    result = {'files': findings, 'status': 'SOURCE_REVIEW_REQUIRED'}
    if not all(x['status'] == 'MATCH' for x in findings):
        return result
    protocol = project / 'revision_outputs/02_FD001_protocol'
    with (protocol / 'engine_splits.csv').open(encoding='utf-8-sig', newline='') as stream:
        split_rows = list(csv.DictReader(stream))
    with (protocol / 'train_evaluation_reference.csv').open(encoding='utf-8-sig', newline='') as stream:
        reference = list(csv.DictReader(stream))
    if len(reference) != 20631:
        raise ValueError('FD001 referans satır sayısı 20631 değil.')
    raw_keys = []
    with (project / 'data/train_FD001.txt').open() as stream:
        for row in stream:
            if not row.strip():
                continue
            values = [float(x) for x in row.split()]
            if len(values) != 26 or not all(math.isfinite(x) for x in values):
                raise ValueError('FD001 satırında sütun sayısı veya sonluluk sorunu.')
            if values[0] != int(values[0]) or values[1] != int(values[1]):
                raise ValueError('Motor/çevrim kimliği tam sayı değil.')
            raw_keys.append((int(values[0]), int(values[1])))
    reference_keys = [(int(float(r['UnitNumber'])), int(float(r['Time']))) for r in reference]
    if raw_keys != reference_keys or len(set(raw_keys)) != len(raw_keys):
        raise ValueError('FD001 veri/referans sırası veya çevrim benzersizliği uyuşmuyor.')
    maxima = {}
    for engine, cycle in raw_keys:
        maxima[engine] = max(maxima.get(engine, 0), cycle)
    engine_keys = {r['engine_key'] for r in reference}
    if len(maxima) != 100 or len(engine_keys) != 100:
        raise ValueError('FD001 motor sayısı 100 değil.')
    engine_key_pairs = {(int(float(r['UnitNumber'])), r['engine_key']) for r in reference}
    if len(engine_key_pairs) != 100:
        raise ValueError('Motor kimliği ile engine_key eşlemesi birebir değil.')
    if len({r['sample_key'] for r in reference}) != len(reference):
        raise ValueError('sample_key benzersiz değil.')
    for r, (engine, cycle) in zip(reference, raw_keys):
        rul = maxima[engine] - cycle
        label = str(r['late_life_le_30']).strip().lower()
        if label not in ('0', '1', '0.0', '1.0', 'true', 'false'):
            raise ValueError('Bilinmeyen RUL etiket gösterimi.')
        positive = label in ('1', '1.0', 'true')
        if float(r['RUL']) != rul or positive != (rul <= 30):
            raise ValueError('RUL referansı ham çevrimlerle uyuşmuyor.')
    cv_rows = [r for r in split_rows if r['scheme'] == 'cv']
    if {int(float(r['fold'])) for r in cv_rows} != set(range(1, 11)):
        raise ValueError('CV fold kümesi 1–10 değil.')
    outer_seen, folds = Counter(), []
    for fold in range(1, 11):
        rows = [r for r in cv_rows if int(float(r['fold'])) == fold]
        roles = {r['engine_key']: r['role'] for r in rows}
        if len(rows) != 100 or set(roles) != engine_keys:
            raise ValueError(f'Fold {fold}: eksik veya yinelenen motor.')
        counts = Counter(roles.values())
        if counts != {'fit': 70, 'early_stop': 10, 'selection': 10, 'outer_valid': 10}:
            raise ValueError(f'Fold {fold}: 70/10/10/10 bölümü uyuşmuyor.')
        outer_seen.update(k for k, v in roles.items() if v == 'outer_valid')
        row_counts = Counter(roles[r['engine_key']] for r in reference)
        eligible = {r['engine_key'] for r in reference
                    if roles[r['engine_key']] == 'fit' and float(r['RUL']) <= 30}
        if len(eligible) < 30:
            raise ValueError(f'Fold {fold}: 30 farklı fit motorundan geç yaşam çevrimi seçilemiyor.')
        folds.append({'fold': fold, 'engine_counts': dict(counts),
                      'row_counts': dict(row_counts),
                      'fit_engines_eligible_for_30_proxy_labels': len(eligible)})
    if set(outer_seen) != engine_keys or any(v != 1 for v in outer_seen.values()):
        raise ValueError('Dış CV motor kapsamı eşleşmiyor.')
    result.update(status='PASS_SOURCES', folds=folds, n_cycles=len(raw_keys), n_engines=100,
                  label_definition='RUL <= 30; model-independent late-life proxy, not annotated anomaly onset')
    return result


def python_candidates(explicit=None):
    candidates = [Path(explicit).expanduser()] if explicit else [Path(sys.executable)]
    if not explicit:
        candidates.append(Path('/opt/anaconda3/envs/cmapss-revision/bin/python'))
        for base in (Path('/opt/anaconda3'), Path('/opt/miniconda3'),
                     Path.home() / 'miniconda3', Path.home() / 'anaconda3'):
            candidates.extend(sorted((base / 'envs').glob('*/bin/python')))
        candidates.extend(sorted((Path.home() / '.venvs').glob('cmapss*/bin/python')))
        candidates.extend(sorted((Path.home() / '.conda/envs').glob('*/bin/python')))
    unique = []
    for path in candidates:

        value = str(path.absolute())
        if path.is_file() and value not in unique:
            unique.append(value)
    return unique


METADATA_PROBE = r'''
import json, platform, sys, importlib.metadata as m
versions = {}
for name in ['torch', 'numpy', 'pandas', 'scikit-learn', 'scipy', 'tensorflow', 'keras']:
    try: versions[name] = m.version(name)
    except m.PackageNotFoundError: versions[name] = None
print('PREFLIGHT_JSON=' + json.dumps(dict(python=platform.python_version(),
    executable=sys.executable, machine=platform.machine(), packages=versions)))
'''

TORCH_PROBE = r'''
import json, time
import numpy as np
import torch
torch.set_num_threads(1)
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)
# CPU smoke check only. Synthetic values; no C-MAPSS training or performance result.
x = torch.from_numpy(np.linspace(-1, 1, 32 * 21, dtype=np.float32).reshape(32, 21))
layer = torch.nn.Linear(21, 4)
loss = layer(x).square().mean()
loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in layer.parameters())
# Same queue dimensions as the published implementation. This is a resource probe,
# not the SEAD-Net network or a timing measurement for a complete training run.
q = torch.nn.Parameter(torch.randn(512, 2))
k = torch.nn.functional.normalize(torch.randn(512, 2), dim=1)
queue = torch.nn.functional.normalize(torch.randn(65536, 2), dim=1)
start = time.perf_counter()
qn = torch.nn.functional.normalize(q, dim=1)
logits = torch.cat([(qn * k).sum(1, keepdim=True), qn @ queue.T], dim=1) / 0.07
nce = torch.nn.functional.cross_entropy(logits, torch.zeros(512, dtype=torch.long))
nce.backward()
assert torch.isfinite(nce) and torch.isfinite(q.grad).all()
elapsed = time.perf_counter() - start
mps = getattr(torch.backends, 'mps', None)
print('PREFLIGHT_JSON=' + json.dumps(dict(status='PASS_TORCH_CPU', torch=torch.__version__,
    numpy=np.__version__, queue_shape=[65536, 2], batch_size=512,
    logits_mib=logits.numel() * logits.element_size() / (1024**2),
    synthetic_nce_forward_backward_seconds=elapsed,
    cuda_available=torch.cuda.is_available(),
    mps_available=bool(mps and mps.is_available()),
    execution_device='cpu', synthetic_check_only=True)))
'''


def probe(executable, code, timeout):
    env = os.environ.copy()
    env['OMP_NUM_THREADS'] = '1'
    env['MKL_NUM_THREADS'] = '1'
    try:
        proc = subprocess.run([executable, '-u', '-X', 'faulthandler', '-c', code],
                              capture_output=True, text=True, timeout=timeout, env=env)
        found = [s[len('PREFLIGHT_JSON='):] for s in proc.stdout.splitlines()
                 if s.startswith('PREFLIGHT_JSON=')]
        return {'returncode': proc.returncode,
                'result': json.loads(found[-1]) if found else None,
                'stderr_tail': proc.stderr[-3000:]}
    except subprocess.TimeoutExpired:
        return {'returncode': None, 'result': None, 'error': 'TIMEOUT'}
    except Exception as exc:
        return {'returncode': None, 'result': None, 'error': repr(exc)}


def check_environment(explicit=None):
    reports, selected = [], None
    for executable in python_candidates(explicit):
        print('Python kontrolü:', executable, flush=True)
        meta = probe(executable, METADATA_PROBE, 20)
        item = {'executable': executable, 'metadata': meta}
        value = meta.get('result')
        if meta.get('returncode') != 0 or not value:
            item['status'] = 'METADATA_FAILED'
        elif not value['packages'].get('torch'):
            item['status'] = 'TORCH_NOT_INSTALLED'
        elif not value['packages'].get('numpy'):
            item['status'] = 'NUMPY_NOT_INSTALLED'
        else:
            print('PyTorch ayrı işlemde test ediliyor…', flush=True)
            test = probe(executable, TORCH_PROBE, 60)
            item['torch_probe'] = test
            if test.get('returncode') == 0 and (test.get('result') or {}).get('status') == 'PASS_TORCH_CPU':
                item['status'] = 'PASS_TORCH_CPU'
                selected = executable
            else:
                item['status'] = 'TORCH_IMPORT_OR_OPERATION_FAILED'
        reports.append(item)
        if selected:
            break
    return {'status': 'PASS_ENVIRONMENT' if selected else 'ENVIRONMENT_REVIEW_REQUIRED',
            'selected_python': selected, 'candidates': reports,
            'note': 'Import/arithmetic preflight only; does not certify a final training environment or cross-framework equivalence.'}


def run_preflight(project, config, explicit_python=None):
    project = Path(project).expanduser().absolute()
    if not project.is_dir():
        raise FileNotFoundError('PROJECT_DIR mevcut klasör olmalı: ' + str(project))
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    out = project / 'revision_outputs/15A_SEAD_preflight' / stamp
    out.mkdir(parents=True, exist_ok=False)
    result = {'schema': 'sead_preflight_v1', 'created_utc': stamp,
              'project': str(project), 'notebook_python': sys.executable,
              'platform': platform.platform(), 'source_audit': config['source_audit'],
              'scientific_results_generated': False, 'existing_results_modified': False}
    try:
        result['sources'] = check_sources(project, config)
    except Exception as exc:
        result['sources'] = {'status': 'SOURCE_REVIEW_REQUIRED', 'error': repr(exc)}
    result['environment'] = check_environment(explicit_python)
    ok = result['sources']['status'] == 'PASS_SOURCES' and result['environment']['status'] == 'PASS_ENVIRONMENT'
    result['status'] = 'PASS_PREFLIGHT' if ok else 'REVIEW_REQUIRED'
    path = out / 'sead_preflight_summary.json'
    path.write_text(json.dumps(result, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    print('\nDURUM:', result['status'])
    print('Veri/protokol:', result['sources']['status'])
    print('Çalışma ortamı:', result['environment']['status'])
    print('GÖNDERİLECEK DOSYA:', path)
    return result


In [ ]:
preflight_result = run_preflight(PROJECT_DIR, CONFIG, PYTHON_EXECUTABLE)
